# Prometheus v0.80: CRLS Learning Loop Demo

**Causal Reinforcement Learning from Self-Correction (CRLS)**

This notebook demonstrates:
1. **EvaluatorAgent** - Post-mortem causal critique of games
2. **CorrectorAgent** - Synthesis of critiques into strategic improvements
3. **Performance Logging** - CSV tracking with live visualization
4. **Observable Learning** - Win rate improvement over 100+ games

The CRLS loop enables agents to learn from mistakes and improve performance through causal reasoning.

In [ ]:
# Imports
import sys
import os
import random
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML, clear_output
import time

# Add prometheus to path
sys.path.insert(0, '/home/pmc/Prometheus_v0_PoC')

from prometheus.game_suite import Connect4, GameResult
from prometheus.evaluator_agent import EvaluatorAgent, CausalCritique
from prometheus.corrector_agent import CorrectorAgent
from prometheus.performance_logger import PerformanceLogger, create_performance_plot
from prometheus.notebook_viz import GameBoardViz

print("✅ Imports successful")
print("\n🎯 CRLS Components Loaded:")
print("  - EvaluatorAgent: Causal critique generation")
print("  - CorrectorAgent: Strategic synthesis")
print("  - PerformanceLogger: CSV logging + live viz")
print("  - Connect4: Game environment")

## Part 1: Define Learning Agent

We'll create a simple learning agent that can incorporate strategic feedback.

In [ ]:
class LearningConnect4Agent:
    """
    Connect4 agent that learns from strategic feedback
    
    Starts with basic heuristics and improves based on
    causal critiques from the CRLS loop.
    """
    
    def __init__(self, agent_id: str = "learner_001"):
        self.agent_id = agent_id
        self.strategy = "Play systematically"
        self.games_played = 0
        
        # Learning parameters
        self.blocking_priority = 0.5  # Increases with feedback
        self.center_preference = 0.3  # Increases with feedback
        self.win_detection = 0.9     # Always high
    
    def update_strategy(self, strategic_prompt: str):
        """Update strategy based on corrector feedback"""
        self.strategy = strategic_prompt
        
        # Parse strategy to update priorities
        prompt_lower = strategic_prompt.lower()
        
        if "critical priority" in prompt_lower and "block" in prompt_lower:
            self.blocking_priority = min(1.0, self.blocking_priority + 0.1)
        
        if "center" in prompt_lower:
            self.center_preference = min(0.8, self.center_preference + 0.1)
    
    def select_move(self, game_state) -> int:
        """Select move using learned heuristics"""
        self.games_played += 1
        
        # Handle both GameState object and dict
        if hasattr(game_state, 'board'):
            board = game_state.board
            current_player = game_state.current_player
        else:
            board = game_state['board']
            current_player = game_state['current_player']
        
        valid_moves = [col for col in range(7) if board[0][col] == 0]
        
        if not valid_moves:
            return None
        
        # 1. Check for winning moves (always prioritize)
        if random.random() < self.win_detection:
            for move in valid_moves:
                if self._would_win(board, move, current_player):
                    return move
        
        # 2. Block opponent wins (learned priority)
        if random.random() < self.blocking_priority:
            opponent = -current_player
            for move in valid_moves:
                if self._would_win(board, move, opponent):
                    return move
        
        # 3. Prefer center columns (learned preference)
        if random.random() < self.center_preference:
            center_moves = [m for m in valid_moves if 2 <= m <= 4]
            if center_moves:
                return random.choice(center_moves)
        
        # 4. Random from remaining
        return random.choice(valid_moves)
    
    def _would_win(self, board, col: int, player: int) -> bool:
        """Check if move would win"""
        # Find where piece would land
        row = None
        for r in range(5, -1, -1):
            if board[r][col] == 0:
                row = r
                break
        
        if row is None:
            return False
        
        # Temporarily place piece
        board[row][col] = player
        win = self._check_win(board, row, col, player)
        board[row][col] = 0
        
        return win
    
    def _check_win(self, board, row: int, col: int, player: int) -> bool:
        """Check for 4-in-a-row"""
        # Horizontal
        count = 1
        for c in range(col - 1, -1, -1):
            if board[row][c] == player:
                count += 1
            else:
                break
        for c in range(col + 1, 7):
            if board[row][c] == player:
                count += 1
            else:
                break
        if count >= 4:
            return True
        
        # Vertical
        count = 1
        for r in range(row + 1, 6):
            if board[r][col] == player:
                count += 1
            else:
                break
        for r in range(row - 1, -1, -1):
            if board[r][col] == player:
                count += 1
            else:
                break
        if count >= 4:
            return True
        
        # Diagonal (down-right)
        count = 1
        r, c = row + 1, col + 1
        while r < 6 and c < 7 and board[r][c] == player:
            count += 1
            r += 1
            c += 1
        r, c = row - 1, col - 1
        while r >= 0 and c >= 0 and board[r][c] == player:
            count += 1
            r -= 1
            c -= 1
        if count >= 4:
            return True
        
        # Diagonal (down-left)
        count = 1
        r, c = row + 1, col - 1
        while r < 6 and c >= 0 and board[r][c] == player:
            count += 1
            r += 1
            c -= 1
        r, c = row - 1, col + 1
        while r >= 0 and c < 7 and board[r][c] == player:
            count += 1
            r -= 1
            c += 1
        if count >= 4:
            return True
        
        return False

print("✅ LearningConnect4Agent defined")
print("\n📚 Agent Capabilities:")
print("  - Adaptive blocking priority")
print("  - Learning center preference")
print("  - Strategic prompt integration")

## Part 2: Define Random Opponent

In [ ]:
class RandomOpponent:
    """Simple random opponent for testing"""
    
    def __init__(self, name: str = "RandomBot"):
        self.name = name
    
    def select_move(self, game_state) -> int:
        """Select random valid move"""
        if hasattr(game_state, 'board'):
            board = game_state.board
        else:
            board = game_state['board']
        
        valid_moves = [col for col in range(7) if board[0][col] == 0]
        return random.choice(valid_moves) if valid_moves else None

print("✅ RandomOpponent defined")

## Part 3: Initialize CRLS Components

In [ ]:
# Initialize CRLS loop components
evaluator = EvaluatorAgent("evaluator_v080")
corrector = CorrectorAgent("corrector_v080")
logger = PerformanceLogger("crls_performance.csv", window_size=20)

# Initialize agents
learning_agent = LearningConnect4Agent("learner_001")
opponent = RandomOpponent("RandomBot")

print("✅ CRLS Loop Initialized")
print("\n🔄 Components:")
print(f"  - Evaluator: {evaluator.agent_id}")
print(f"  - Corrector: {corrector.agent_id}")
print(f"  - Logger: crls_performance.csv")
print(f"  - Agent: {learning_agent.agent_id}")
print(f"  - Opponent: {opponent.name}")

## Part 4: Run CRLS Learning Loop

We'll run 100 games with the CRLS loop:
1. Play games
2. Evaluate performance (EvaluatorAgent)
3. Synthesize improvements (CorrectorAgent)
4. Update agent strategy
5. Repeat

In [ ]:
def play_game_with_logging(agent, opponent, agent_player: int, generation: int):
    """
    Play single game and return history
    """
    game = Connect4()
    moves_made = []
    
    while game.result.value == 'ongoing':
        current_player = game.current_player
        
        # Get move from appropriate player
        if current_player == agent_player:
            move = agent.select_move(game.get_state())
        else:
            move = opponent.select_move(game.get_state())
        
        # Make move
        if move is not None:
            success = game.make_move(move)
            if success:
                moves_made.append((current_player, move))
            else:
                break
        else:
            break
    
    # Create game history
    game_history = {
        'moves': moves_made,
        'result': game.result.value,
        'winner': game.winner,
        'agent_player': agent_player
    }
    
    # Log to CSV
    logger.log_game(game_history, agent_player, generation)
    
    return game_history


# Run CRLS learning loop
print("🚀 Starting CRLS Learning Loop...")
print("="*60)

NUM_GAMES = 100
GAMES_PER_BATCH = 10  # Evaluate and update every 10 games

for batch_num in range(NUM_GAMES // GAMES_PER_BATCH):
    batch_histories = []
    batch_agent_players = []
    
    # Play batch of games
    for game_num in range(GAMES_PER_BATCH):
        # Alternate who goes first
        agent_player = 1 if game_num % 2 == 0 else -1
        
        # Play game
        game_history = play_game_with_logging(
            learning_agent, opponent, agent_player, batch_num
        )
        
        batch_histories.append(game_history)
        batch_agent_players.append(agent_player)
    
    # CRLS Loop: Evaluate → Correct → Update
    
    # 1. Evaluate games
    critiques = evaluator.evaluate_multiple_games(batch_histories, batch_agent_players)
    
    # 2. Synthesize strategy
    strategic_prompt = corrector.synthesize_strategy(critiques)
    
    # 3. Update agent
    learning_agent.update_strategy(strategic_prompt)
    
    # Print progress
    stats = logger.get_stats()
    wins_in_batch = sum(1 for h in batch_histories if h['winner'] == h['agent_player'])
    
    print(f"\nBatch {batch_num + 1}/{NUM_GAMES // GAMES_PER_BATCH}:")
    print(f"  Games {batch_num * GAMES_PER_BATCH + 1}-{(batch_num + 1) * GAMES_PER_BATCH}")
    print(f"  Batch Win Rate: {wins_in_batch}/{GAMES_PER_BATCH} = {wins_in_batch/GAMES_PER_BATCH:.1%}")
    print(f"  Rolling Win Rate: {stats['current_rolling_win_rate']:.1%}")
    print(f"  Overall Win Rate: {stats['overall_win_rate']:.1%}")
    print(f"  Agent Priorities: Block={learning_agent.blocking_priority:.2f}, Center={learning_agent.center_preference:.2f}")
    
    # Show latest strategic prompt
    if batch_num % 2 == 0:  # Every other batch
        print(f"\n  📋 Latest Strategy:")
        print(f"     {strategic_prompt[:200]}...")

print("\n" + "="*60)
print("✅ CRLS Learning Loop Complete!")
print(f"\nFinal Statistics:")
final_stats = logger.get_stats()
print(f"  Total Games: {final_stats['total_games']}")
print(f"  Overall Win Rate: {final_stats['overall_win_rate']:.1%}")
print(f"  Final Rolling Win Rate: {final_stats['current_rolling_win_rate']:.1%}")
print(f"\nAgent Learning:")
print(f"  Blocking Priority: {learning_agent.blocking_priority:.2f}")
print(f"  Center Preference: {learning_agent.center_preference:.2f}")

## Part 5: Visualize Learning Performance

In [ ]:
# Create performance plot
fig = create_performance_plot("crls_performance.csv")
plt.show()

print("\n📊 Performance visualization complete!")
print("\nThe plot shows:")
print("  Top: Rolling win rate over time (20-game window)")
print("  Bottom: Win rate by generation/batch")
print("\n✨ Observable learning through CRLS loop!")

## Part 6: Examine Causal Critiques

In [ ]:
# Show example critiques from recent games
games = logger.read_log()
recent_games = games[-5:]  # Last 5 games

print("🔍 Recent Causal Critiques:")
print("="*60)

for i, game in enumerate(recent_games, 1):
    # Recreate game history for evaluation
    game_hist = {
        'moves': [],  # We don't have full move history in CSV
        'result': game['result'],
        'winner': game['winner'],
        'agent_player': game['agent_player']
    }
    
    critique = evaluator.evaluate_game(game_hist, game['agent_player'])
    
    print(f"\nGame {game['game_number']}:")
    print(f"  Result: {critique.game_result.upper()}")
    print(f"  Critique: {critique.reason}")
    if critique.alternative_move is not None:
        print(f"  Suggested Move: Column {critique.alternative_move}")
    print(f"  Confidence: {critique.confidence:.1%}")

print("\n" + "="*60)

## Part 7: Strategy Evolution Timeline

In [ ]:
# Show how strategy evolved over time
strategy_history = corrector.get_strategy_evolution()

print("📈 Strategy Evolution Timeline:")
print("="*60)

# Show every 2nd generation to avoid clutter
for i, entry in enumerate(strategy_history[::2]):
    print(f"\nGeneration {entry['generation']}:")
    print(f"  Win Rate: {entry['win_rate']:.1%}")
    print(f"  Strategy:")
    # Show first 300 chars of strategy
    strategy_preview = entry['strategy'][:300]
    print(f"    {strategy_preview}...")

print("\n" + "="*60)
print("\n✅ CRLS Loop demonstrates:")
print("  1. Causal evaluation of game outcomes")
print("  2. Strategic synthesis from critiques")
print("  3. Measurable performance improvement")
print("  4. Learning from self-correction")

## Summary: v0.80 CRLS Loop

**What we've demonstrated:**

1. **EvaluatorAgent** - Post-mortem causal analysis identifying:
   - Blocking failures
   - Missed winning opportunities
   - Strategic mistakes

2. **CorrectorAgent** - Strategic synthesis that:
   - Aggregates multiple critiques
   - Prioritizes most critical improvements
   - Generates actionable strategic prompts

3. **Performance Logging** - Complete tracking with:
   - CSV logging of all games
   - Rolling win rate calculation
   - Live visualization support

4. **Observable Learning** - Measurable improvement through:
   - Win rate increase over time
   - Adaptive parameter tuning
   - Strategic evolution

**Key Innovation:** The CRLS loop enables goal-directed learning by identifying causal relationships between actions and outcomes, then synthesizing targeted improvements.

**Next Steps (v0.85):**
- Hofstadter's Analogy for strategy transfer
- MCS Alignment Governor for safety
- Multi-game learning